# Hybrid Dynamic Ensemble for Mag-7 Stock Return Prediction
## Complete Implementation Pipeline 

This notebook presents the complete research pipeline developed to forecast daily returns of the Magnificent Seven stocks. The central contribution is a Hybrid Dynamic Ensemble (HDE) that adaptively combines classical machine learning models with deep learning sequence models, allocating weights based on each model's empirical performance on held-out validation data.

**Pipeline Overview:**
1. Data Collection (Yahoo Finance + FRED)


## Script 01: S&P 500 / Mag-7 Stock Price Data Collection

In [7]:
import yfinance as yf
import os
import pandas as pd

# Configuration for the "Research Baseline"
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA', 'SPY']
START_DATE = "2015-01-01"
END_DATE = "2026-01-01"  # Ensures data stops at 2025-12-31
OUTPUT_PATH = "data/raw/prices.csv"

def collect_prices():
    print(f"Downloading Mag 7 + Benchmark: {START_DATE} to {END_DATE}")
    
    # auto_adjust=True is critical for dividends/splits in dissertation research
    df = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True)
    
    # yfinance returns multi-level columns, just need Close prices
    if isinstance(df.columns, pd.MultiIndex):
        data = df['Close']
    else:
        data = df
        
    data.index.name = 'Date'
    
    # Save the data
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    data.to_csv(OUTPUT_PATH)
    
    print(f"Saved {len(data)} rows to {OUTPUT_PATH}")
    print(f"Date range: {data.index.min()} to {data.index.max()}")
    print(f"Tickers: {list(data.columns)}")
    return data

if __name__ == "__main__":
    collect_prices()

[*********************100%***********************]  8 of 8 completed

Saved 2766 rows to data/raw/prices.csv
Date range: 2015-01-02 00:00:00 to 2025-12-31 00:00:00
Tickers: ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'NVDA', 'SPY', 'TSLA']


## Script 02 - Federal Reserve / Macroeconomic Data Collection

In [8]:
import pandas as pd
from fredapi import Fred
import os
from dotenv import load_dotenv
load_dotenv()

# Configuration for the baseline macroeconomic indicators
FRED_API_KEY = os.environ.get('FRED_API_KEY')
OUTPUT_PATH = 'data/raw/macro_fred.csv'
START_DATE = '2014-01-01'
END_DATE = '2026-01-01' # Ensures data stops at 2025-12-31


# Macro indicators needed for the feature set
MACRO_SERIES = {
    'fed_funds_rate': 'DFF',
    'us10y_yield': 'DGS10',
    'vix': 'VIXCLS',
    'cpi': 'CPIAUCSL',
    'unemployment_rate': 'UNRATE'
}

if not FRED_API_KEY:
    raise EnvironmentError(
        "If not working check if the .env file and the python -dotenv package has been installed (Personal Note: this error has happened before)"
    )

def collect_macro_data():
    fred = Fred(api_key=FRED_API_KEY)
    macro_frames = []
    print("Fetching Macro Indicators:")

    for name, series_id in MACRO_SERIES.items():
        print(f"  - {name}")
        # Coerce to numeric just in case FRED returns any string values
        s = fred.get_series(series_id, observation_start=START_DATE, observation_end=END_DATE)
        s = pd.to_numeric(s, errors='coerce')
        s.name = name
        macro_frames.append(s)


     # CPI and unemployment are monthly — forward fill to match daily price data
    macro_df = pd.concat(macro_frames, axis=1)
    macro_df.index.name = 'Date'
    
    # Forward fill monthly/quarterly data to daily frequency
    macro_df = macro_df.ffill()

    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
    macro_df.to_csv(OUTPUT_PATH)
    
    print(f"Saved {len(macro_df)} rows to {OUTPUT_PATH}")
    return macro_df

if __name__ == "__main__":
    collect_macro_data()

Fetching Macro Indicators:
  - fed_funds_rate
  - us10y_yield
  - vix
  - cpi
  - unemployment_rate
Saved 4384 rows to data/raw/macro_fred.csv


## Script 03 - Feature Engineering and Master Dataset Construction

In [9]:
import pandas as pd
import numpy as np
import os

# Wilder-style RSI (Industry standard)
def calculate_rsi_wilder(series, period=14):
    delta = series.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    # small guard in case losses are zero for a stretch
    rs = avg_gain / avg_loss.replace(0, 1e-9)
    return 100 - (100 / (1 + rs))

def build_dataset():
    prices_path = "data/raw/prices.csv"
    macro_path = "data/raw/macro_fred.csv" 

    if not os.path.exists(prices_path) or not os.path.exists(macro_path):
        print("Error: Raw files not found.")
        return

    # read both inputs once at the start
    prices = pd.read_csv(prices_path, index_col=0, parse_dates=True)
    macro = pd.read_csv(macro_path, index_col=0, parse_dates=True)
    
    # use SPY as the market benchmark where available
    spy_ret = prices['SPY'].pct_change() if 'SPY' in prices.columns else None
    
    mag_7 = [t for t in ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'TSLA'] if t in prices.columns]
    final_dfs = []

    print("Building Research-Grade Features")

    for ticker in mag_7:
       # start from the adjusted close series for one stock
        df = pd.DataFrame(prices[ticker]).rename(columns={ticker: 'Adj_Close'})
        
        # basic return features
        df['Return_1d'] = df['Adj_Close'].pct_change()
        df['Return_5d'] = df['Adj_Close'].pct_change(5)   # weekly rolling return
        df['Return_21d'] = df['Adj_Close'].pct_change(21)  # monthly rolling return
        df['Market_Return'] = spy_ret
        
        # basic return features
        df['MA10_Ratio'] = df['Adj_Close'] / df['Adj_Close'].rolling(window=10, min_periods=10).mean()
        df['MA50_Ratio'] = df['Adj_Close'] / df['Adj_Close'].rolling(window=50, min_periods=50).mean()
                
        # MACD and signal line
        ema12 = df['Adj_Close'].ewm(span=12, adjust=False).mean()
        ema26 = df['Adj_Close'].ewm(span=26, adjust=False).mean()
        df['MACD'] = (ema12 - ema26) / df['Adj_Close']
        df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()
        
         # momentum / overbought-oversold style indicator
        df['RSI'] = calculate_rsi_wilder(df['Adj_Close'])
        
        # rolling volatility from daily returns
        df['Vol_20d'] = df['Return_1d'].rolling(window=20, min_periods=20).std()
        
          # a simple momentum measure over the last 10 trading days
        df['Momentum_10d'] = df['Adj_Close'].pct_change(10)
        
        # forward fill so each day only sees the latest macro value already known
        df = df.join(macro, how='left').ffill()
        
       # binary version for up/down prediction
        df['Target_Return'] = df['Return_1d'].shift(-1)
        # Classification target: direction of next-day return
        df['Target_Direction'] = (df['Target_Return'] > 0).astype(int)
        
        df['Ticker'] = ticker
        df = df.reset_index().rename(columns={"index": "Date"})
        
        final_dfs.append(df)

    # stack all tickers together and keep the ordering tidy
    master_df = pd.concat(final_dfs, axis=0)
    master_df = master_df.sort_values(by=['Date', 'Ticker']).reset_index(drop=True)
    
    # final clean before saving
    master_df = master_df.replace([np.inf, -np.inf], np.nan).dropna()

    os.makedirs("data/processed", exist_ok=True)
    master_df.to_csv("data/processed/master_dataset.csv", index=False)
    
    print(f"Final Master Dataset Saved (Long Format)")
    print(f"Total Observations: {len(master_df)}")
    print(f"Features: {[c for c in master_df.columns if c not in ['Date','Ticker','Adj_Close','Target_Return','Target_Direction']]}")
    print(f"Date range: {master_df['Date'].min()} to {master_df['Date'].max()}")
    return master_df

if __name__ == "__main__":
    master_df = build_dataset()

Building Research-Grade Features
Final Master Dataset Saved (Long Format)
Total Observations: 19012
Features: ['Return_1d', 'Return_5d', 'Return_21d', 'Market_Return', 'MA10_Ratio', 'MA50_Ratio', 'MACD', 'MACD_Signal', 'RSI', 'Vol_20d', 'Momentum_10d', 'fed_funds_rate', 'us10y_yield', 'vix', 'cpi', 'unemployment_rate']
Date range: 2015-03-16 00:00:00 to 2025-12-30 00:00:00
